# 07 — ML Condition Model Validation

## Purpose

Validate the controlled ML relationship for estimating observed `zustandsnote` using:
- latitude
- longitude
- DTV
- Bauwerkstoff
- optionally Bauwerksart

`Länge` and `Breite` remain excluded from this condition-model validation.

This notebook is a validation stage, not the frozen production model and not a structural design/FEM calculation.


In [1]:
from getpass import getpass
from pathlib import Path
import os
import json
import warnings

def find_project_root():
    env_root = os.getenv("BRIDGE_PROJECT_ROOT")
    if env_root:
        root = Path(env_root).expanduser().resolve()
        if (root / "Dataset_PlanA-B").exists():
            return root
        raise FileNotFoundError(
            f"BRIDGE_PROJECT_ROOT does not contain Dataset_PlanA-B: {root}"
        )
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "Dataset_PlanA-B").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Set BRIDGE_PROJECT_ROOT to the project folder."
    )

PROJECT_ROOT = find_project_root()
DATASET_ROOT = PROJECT_ROOT / "Dataset_PlanA-B"
OUTPUT_ROOT = PROJECT_ROOT / "Output_PlanA-B"
OUTPUT_DIR = OUTPUT_ROOT / "07_ML_Condition_Model_Validation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "Final_Project"
DB_USER = "postgres"
DB_PASSWORD = getpass("PostgreSQL password: ")

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)
print("Output dir :", OUTPUT_DIR)


Project root: C:\Datenanalyse\final Project
Dataset root: C:\Datenanalyse\final Project\Dataset_PlanA-B
Output dir : C:\Datenanalyse\final Project\Output_PlanA-B\07_ML_Condition_Model_Validation


## 00A — DATA SOURCE / INPUT–OUTPUT MANIFEST

| Item | Source / origin | Transfer method | Role in Notebook 07 | Destination |
|---|---|---|---|---|
| Condition dataset | PostgreSQL `Final_Project` → `final.bridge_ml_dataset_final` | SQL query | Canonical ML validation input | In-memory `df` |
| Geometry | `geom_x` / `geom_y` | SQL + coordinate transform | Latitude/longitude predictors | `work` |
| Traffic | `traffic_dtv_mean` | SQL | DTV predictor | `work` |
| Material | `baustoffklasse` | SQL | Material predictor | `work` |
| Bridge type | `bauwerksart_text` | SQL | Optional predictor / candidate variable | `work` |
| Target | `zustandsnote` | SQL | Regression target | `work` |
| Validation outputs | Model/dataframe results | Local file write | Audit/review | `Output_PlanA-B/07_ML_Condition_Model_Validation` |

### Transfer chain

```text
Notebook 06
    ↓
PostgreSQL: final.bridge_ml_dataset_final
    ↓
Notebook 07
    ├── controlled feature preparation
    ├── baseline model
    ├── full model
    ├── per-type validation
    ├── condition distribution
    └── candidate-type comparison
    ↓
Output_PlanA-B/07_ML_Condition_Model_Validation
```

Notebook 07 does not download BASt/DWD/Traffic data, does not perform imputation as a separate pipeline step, and does not modify the frozen production model.


## 01 — Load canonical PostgreSQL dataset

Use the same canonical table already validated in Notebook 07/08.


In [2]:
# 01 — Load canonical PostgreSQL dataset

from sqlalchemy import create_engine, URL, text
import pandas as pd
import numpy as np
from pyproj import Transformer

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

SOURCE_TABLE = '"final"."bridge_ml_dataset_final"'

url = URL.create(
    "postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)
engine = create_engine(url, connect_args={"connect_timeout": 10})

with engine.connect() as conn:
    print("PostgreSQL:", conn.execute(text("SELECT current_database()")).scalar())

df = pd.read_sql(f"SELECT * FROM {SOURCE_TABLE}", engine)

print("Source table:", SOURCE_TABLE)
print("Shape:", df.shape)
print("Columns:", len(df.columns))


PostgreSQL: Final_Project
Source table: "final"."bridge_ml_dataset_final"
Shape: (52214, 97)
Columns: 97


## 02 — Build the controlled analysis table

The coordinate transformation follows the established project representation:

`geom_x / geom_y → EPSG:4326`.

The target is the observed `zustandsnote`.


In [3]:
required = [
    "bridge_id",
    "geom_x",
    "geom_y",
    "traffic_dtv_mean",
    "baustoffklasse",
    "bauwerksart_text",
    "zustandsnote",
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise RuntimeError(f"Missing required columns: {missing}")

tr = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)

x = pd.to_numeric(df["geom_x"], errors="coerce")
y = pd.to_numeric(df["geom_y"], errors="coerce")
lon, lat = tr.transform(x.to_numpy(dtype=float), y.to_numpy(dtype=float))

work = pd.DataFrame({
    "bridge_id": df["bridge_id"].astype(str),
    "latitude": lon * 0 + lat,
    "longitude": lon,
    "dtv": pd.to_numeric(df["traffic_dtv_mean"], errors="coerce"),
    "bauwerkstoff": df["baustoffklasse"].astype("string").str.strip(),
    "bauwerksart": df["bauwerksart_text"].astype("string").str.strip(),
    "zustandsnote": pd.to_numeric(df["zustandsnote"], errors="coerce"),
})

work = work.dropna().copy()
work = work[
    work["zustandsnote"].between(1, 4)
    & work["latitude"].between(-90, 90)
    & work["longitude"].between(-180, 180)
].copy()

print("Usable rows:", f"{len(work):,}")
print("Bridge types:", work["bauwerksart"].nunique())
print("Materials:", work["bauwerkstoff"].nunique())


Usable rows: 52,214
Bridge types: 43
Materials: 7


## 03 — Baseline model without Bauwerksart

This is the control model.

Inputs:

`latitude + longitude + DTV + Bauwerkstoff`

Target:

`zustandsnote`


In [4]:
BASE_FEATURES = ["latitude", "longitude", "dtv", "bauwerkstoff"]
FULL_FEATURES = BASE_FEATURES + ["bauwerksart"]
TARGET = "zustandsnote"

groups = work["bridge_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)
train_idx, test_idx = next(
    gss.split(work[BASE_FEATURES], work[TARGET], groups=groups)
)

Xb_train = work.iloc[train_idx][BASE_FEATURES]
Xb_test = work.iloc[test_idx][BASE_FEATURES]
y_train = work.iloc[train_idx][TARGET]
y_test = work.iloc[test_idx][TARGET]


In [5]:
def make_regression_pipeline(features):
    numeric = [c for c in features if c in ["latitude", "longitude", "dtv"]]
    categorical = [c for c in features if c in ["bauwerkstoff", "bauwerksart"]]

    transformers = []

    if numeric:
        transformers.append(
            (
                "num",
                SimpleImputer(strategy="median"),
                numeric,
            )
        )

    if categorical:
        transformers.append(
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]),
                categorical,
            )
        )

    pre = ColumnTransformer(transformers)

    model = ExtraTreesRegressor(
        n_estimators=400,
        min_samples_leaf=5,
        max_features=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    return Pipeline([
        ("preprocess", pre),
        ("model", model),
    ])

baseline = make_regression_pipeline(BASE_FEATURES)
baseline.fit(Xb_train, y_train)
base_pred = baseline.predict(Xb_test)

baseline_metrics = {
    "MAE": mean_absolute_error(y_test, base_pred),
    "RMSE": mean_squared_error(y_test, base_pred) ** 0.5,
    "R2": r2_score(y_test, base_pred),
}

print("Baseline metrics:")
print(baseline_metrics)


Baseline metrics:
{'MAE': 0.2926114566220836, 'RMSE': 0.3894254671556612, 'R2': 0.2600920418426478}


## 04 — Full model with Bauwerksart

This model adds `Bauwerksart` while keeping exactly the same train/test split.

The difference between the baseline and full model is therefore attributable to adding the bridge-type information to the feature set, within this validation design.


In [6]:
Xf_train = work.iloc[train_idx][FULL_FEATURES]
Xf_test = work.iloc[test_idx][FULL_FEATURES]

full_model = make_regression_pipeline(FULL_FEATURES)
full_model.fit(Xf_train, y_train)
full_pred = full_model.predict(Xf_test)

full_metrics = {
    "MAE": mean_absolute_error(y_test, full_pred),
    "RMSE": mean_squared_error(y_test, full_pred) ** 0.5,
    "R2": r2_score(y_test, full_pred),
}

comparison = pd.DataFrame([
    {"model": "Baseline — without Bauwerksart", **baseline_metrics},
    {"model": "Full — with Bauwerksart", **full_metrics},
])

comparison["MAE_change_vs_baseline"] = (
    comparison["MAE"] - baseline_metrics["MAE"]
)

display(comparison)


,model,MAE,RMSE,R2,MAE_change_vs_baseline
0,Baseline — without Bauwerksart,0.292611,0.389425,0.260092,0.000000
1,Full — with Bauwerksart,0.289030,0.384660,0.278090,-0.003581


## 05 — Per-type validation

The actual observed `Bauwerksart` is used to stratify the test errors.

This evaluates where the model performs well or poorly; it is not a ranking of bridge types.


In [7]:
test_result = work.iloc[test_idx][
    ["bridge_id", "bauwerksart", "bauwerkstoff", "zustandsnote"]
].copy()

test_result["predicted_zustandsnote"] = np.clip(full_pred, 1, 4)
test_result["absolute_error"] = (
    test_result["zustandsnote"] - test_result["predicted_zustandsnote"]
).abs()

per_type = (
    test_result.groupby("bauwerksart")
    .agg(
        n=("bridge_id", "size"),
        MAE=("absolute_error", "mean"),
        mean_observed=("zustandsnote", "mean"),
        mean_predicted=("predicted_zustandsnote", "mean"),
    )
    .sort_values("n", ascending=False)
)

display(per_type)


,n,MAE,mean_observed,mean_predicted
bauwerksart,,,,
Plattenbrücke,3073,0.279275,2.245037,2.238870
"Plattenbalkenbrücke, Trägerrostbrücke",2685,0.270185,2.337989,2.355413
Brücke als offener Rahmen,1201,0.317289,1.933472,1.935836
Brücke als geschlossener Rahmen,1174,0.306940,2.015673,2.024092
Hohlkastenbrücke,474,0.249002,2.531013,2.537145
Balkenbrücke / Mittelträger / Trapezplatte,464,0.261125,2.262931,2.245736
"Rohr als Brücke, ohne Ummantelung",371,0.374544,1.897035,1.923184
Gewölbe-/Bogenbrücke ohne Aufbeton,255,0.319325,2.170980,2.201627
Brücke mit Balken- / Plattenmischsystem,148,0.321858,2.289189,2.320500


## 06 — Condition classes by bridge type

Official project condition intervals:

- 1.0–1.4
- 1.5–1.9
- 2.0–2.4
- 2.5–2.9
- 3.0–3.4
- 3.5–4.0

The table describes the observed historical population.


In [8]:
def condition_class(v):
    if 1.0 <= v <= 1.4:
        return "1.0–1.4"
    if 1.5 <= v <= 1.9:
        return "1.5–1.9"
    if 2.0 <= v <= 2.4:
        return "2.0–2.4"
    if 2.5 <= v <= 2.9:
        return "2.5–2.9"
    if 3.0 <= v <= 3.4:
        return "3.0–3.4"
    if 3.5 <= v <= 4.0:
        return "3.5–4.0"
    return pd.NA

observed = work.copy()
observed["condition_class"] = observed["zustandsnote"].map(condition_class)

condition_distribution = pd.crosstab(
    observed["bauwerksart"],
    observed["condition_class"],
    normalize="index",
) * 100

display(condition_distribution.round(2))


condition_class,1.0–1.4,1.5–1.9,2.0–2.4,2.5–2.9,3.0–3.4,3.5–4.0
bauwerksart,,,,,,
Balkenbrücke / Mittelträger / Trapezplatte,4.45,11.37,58.46,22.82,2.86,0.04
Behelfsbrücke als Walzträgerbrücke,100.00,0.00,0.00,0.00,0.00,0.00
"Behelfsbrücke, Sonstiges System",16.67,16.67,33.33,33.33,0.00,0.00
Bogenartige Brücke / Gewölbebrücke,0.00,0.00,100.00,0.00,0.00,0.00
Bogenbrücke als Mischsystem,5.80,7.25,44.93,37.68,2.90,1.45
Bogenbrücke mit Bogenscheiben,2.50,10.00,40.00,35.00,12.50,0.00
Bogenbrücke mit abgehängter Fahrbahn,4.07,14.63,47.15,23.58,8.94,1.63
Bogenbrücke mit aufgeständerter Fahrbahn,4.92,7.10,51.91,30.60,4.92,0.55
Brücke,0.00,0.00,100.00,0.00,0.00,0.00


## 07 — Candidate-type comparison on the validation model

For a defined new-project scenario, the same project inputs are held constant while `Bauwerksart` is varied.

The resulting values are **model-estimated condition notes**. They are not observed counterfactual measurements and are not structural design results.


In [9]:
def evaluate_candidates(model, latitude, longitude, dtv, bauwerkstoff, candidate_types):
    candidate_df = pd.DataFrame({
        "latitude": latitude,
        "longitude": longitude,
        "dtv": dtv,
        "bauwerkstoff": bauwerkstoff,
        "bauwerksart": list(candidate_types),
    })

    candidate_df["predicted_zustandsnote"] = np.clip(
        model.predict(candidate_df[FULL_FEATURES]),
        1,
        4,
    )

    return candidate_df.sort_values(
        "predicted_zustandsnote"
    ).reset_index(drop=True)

NEW_PROJECT = {
    "latitude": 50.7374,
    "longitude": 7.0982,
    "dtv": 25000.0,
    "bauwerkstoff": "Stahlbeton",
}

candidate_types = (
    work.groupby("bauwerksart")
    .size()
    .loc[lambda s: s >= 100]
    .index
    .tolist()
)

candidate_results = evaluate_candidates(
    full_model,
    **NEW_PROJECT,
    candidate_types=candidate_types,
)

display(candidate_results)


,latitude,longitude,dtv,bauwerkstoff,bauwerksart,predicted_zustandsnote
0,50.7374,7.0982,25000.0,Stahlbeton,"Rohr als Brücke, ohne Ummantelung",1.786593
1,50.7374,7.0982,25000.0,Stahlbeton,Brücke als offener Rahmen,1.986941
2,50.7374,7.0982,25000.0,Stahlbeton,Brücke als Schrägstielrahmen,1.997507
3,50.7374,7.0982,25000.0,Stahlbeton,Brücke als Rahmen-Mischsystem,2.004364
4,50.7374,7.0982,25000.0,Stahlbeton,"Rohr als Brücke, mit Ummantelung",2.088934
5,50.7374,7.0982,25000.0,Stahlbeton,Brücke als geschlossener Rahmen,2.128184
6,50.7374,7.0982,25000.0,Stahlbeton,Plattenbrücke,2.153118
7,50.7374,7.0982,25000.0,Stahlbeton,Rahmenbrücke als Trog-Haube-Konstruktion,2.182509
8,50.7374,7.0982,25000.0,Stahlbeton,Gewölbe- bzw. Bogenbrücke,2.198041
9,50.7374,7.0982,25000.0,Stahlbeton,Balkenbrücke / Mittelträger / Trapezplatte,2.204530


## 08 — Model diagnostic

This final diagnostic reports the relationship between observed and predicted `zustandsnote` for the held-out test set.


In [10]:
diagnostic = test_result[
    ["zustandsnote", "predicted_zustandsnote"]
].copy()

diagnostic["residual"] = (
    diagnostic["zustandsnote"] -
    diagnostic["predicted_zustandsnote"]
)

display(diagnostic.describe())


,zustandsnote,predicted_zustandsnote,residual
count,10443.000000,10443.000000,10443.000000
mean,2.203438,2.207829,-0.004391
std,0.452748,0.234243,0.384653
min,1.000000,1.221875,-1.681030
25%,2.000000,2.051062,-0.211431
50%,2.200000,2.220031,0.000776
75%,2.500000,2.371962,0.225599
max,4.000000,3.001137,1.681133


## 09 — Export validation results


In [11]:
# 09 — Export validation results and audit artifacts

comparison.to_csv(
    OUTPUT_DIR / "07_baseline_vs_bauwerksart_model.csv",
    index=False, encoding="utf-8-sig"
)
per_type.to_csv(
    OUTPUT_DIR / "07_per_type_validation.csv",
    encoding="utf-8-sig"
)
condition_distribution.to_csv(
    OUTPUT_DIR / "07_condition_distribution_by_bridge_type.csv",
    encoding="utf-8-sig"
)
candidate_results.to_csv(
    OUTPUT_DIR / "07_candidate_type_comparison.csv",
    index=False, encoding="utf-8-sig"
)

data_inventory = pd.DataFrame({
    "column_order": range(1, len(work.columns) + 1),
    "column": work.columns,
    "dtype": [str(work[c].dtype) for c in work.columns],
    "missing_count": [int(work[c].isna().sum()) for c in work.columns],
    "role": [
        "target" if c == TARGET else
        "group_id" if c == "bridge_id" else
        "predictor" if c in FULL_FEATURES else
        "derived_or_validation"
        for c in work.columns
    ],
})
data_inventory.to_csv(
    OUTPUT_DIR / "07_analysis_data_inventory.csv",
    index=False, encoding="utf-8-sig"
)

manifest = {
    "stage": 7,
    "notebook": "07_ML_Condition_Model_Validation",
    "source": SOURCE_TABLE,
    "target": TARGET,
    "baseline_features": BASE_FEATURES,
    "full_features": FULL_FEATURES,
    "excluded_from_condition_model": ["laenge", "breite", "FEM", "InfoCAD"],
    "split": "GroupShuffleSplit by bridge_id",
    "purpose": "Validate the controlled contribution of Bauwerksart to condition estimation.",
    "baseline_metrics": baseline_metrics,
    "full_metrics": full_metrics,
}
(OUTPUT_DIR / "07_data_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

human_manifest = (
    "Notebook 07 — ML Condition Model Validation\n"
    "=============================================\n\n"
    f"PROJECT_ROOT: {PROJECT_ROOT}\n"
    f"SOURCE: PostgreSQL {SOURCE_TABLE}\n"
    f"OUTPUT_DIR: {OUTPUT_DIR}\n\n"
    f"TARGET: {TARGET}\n"
    f"BASELINE_FEATURES: {BASE_FEATURES}\n"
    f"FULL_FEATURES: {FULL_FEATURES}\n"
    "EXCLUDED: laenge, breite, FEM, InfoCAD\n"
    "VALIDATION: GroupShuffleSplit by bridge_id\n"
    "PURPOSE: Validate the controlled contribution of Bauwerksart to condition estimation.\n"
    "This notebook does not modify the frozen production model and does not perform FEM/design.\n\n"
    "OUTPUTS:\n"
    "- 07_baseline_vs_bauwerksart_model.csv\n"
    "- 07_per_type_validation.csv\n"
    "- 07_condition_distribution_by_bridge_type.csv\n"
    "- 07_candidate_type_comparison.csv\n"
    "- 07_analysis_data_inventory.csv\n"
    "- 07_data_manifest.json\n"
    "- 07_data_manifest.txt\n"
)
(OUTPUT_DIR / "07_data_manifest.txt").write_text(
    human_manifest, encoding="utf-8"
)

print("07 STATUS: COMPLETE")
print("Output directory:", OUTPUT_DIR)
print("Data manifest:", OUTPUT_DIR / "07_data_manifest.txt")


07 STATUS: COMPLETE
Output directory: C:\Datenanalyse\final Project\Output_PlanA-B\07_ML_Condition_Model_Validation
Data manifest: C:\Datenanalyse\final Project\Output_PlanA-B\07_ML_Condition_Model_Validation\07_data_manifest.txt
